# 1. Initialisation de la session Spark

In [1]:
# initialisation de la session Spark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("NetSentinel - Batch Analysis").config("spark.driver.memory", "8g").getOrCreate()
print(spark.version)

3.5.5


26/03/20 11:04:52 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


# 2. Concaténation des différents .csv

In [2]:
# importation de tous les .csv
import os

data_path = "../data"
df = spark.read.csv(data_path, header=True, inferSchema=True)

print(f"Nombre de lignes : {df.count()}")
print(f"Nombre de colonnes : {len(df.columns)}")

df.printSchema()

Nombre de lignes : 2608247
Nombre de colonnes : 122
root
 |-- flow_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- src_ip: string (nullable = true)
 |-- src_port: string (nullable = true)
 |-- dst_ip: string (nullable = true)
 |-- dst_port: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- packets_count: string (nullable = true)
 |-- fwd_packets_count: string (nullable = true)
 |-- bwd_packets_count: string (nullable = true)
 |-- total_payload_bytes: string (nullable = true)
 |-- fwd_total_payload_bytes: string (nullable = true)
 |-- bwd_total_payload_bytes: string (nullable = true)
 |-- payload_bytes_max: string (nullable = true)
 |-- payload_bytes_min: string (nullable = true)
 |-- payload_bytes_mean: string (nullable = true)
 |-- payload_bytes_std: string (nullable = true)
 |-- payload_bytes_variance: string (nullable = true)
 |-- fwd_payload_bytes_max: double (nullable = true)
 |-- fwd_payload_by

### Vérification du déséquilibre des classes

- Le dataset source BCCC-CIC-IDS-2017 contient 2.6M de connexions réseau réparties sur 15 classes, Après analyse de la distribution des labels, nous avons décidé de filtrer le dataset pour ne garder que les 4 classes les plus représentées, afin de limiter le déséquilibre entre classes tout en conservant les types d'attaques les plus courants en entreprise

- Classes conservées :
    - Benign (trafic normal)
    - DoS_Hulk (attaque par déni de service)
    - Port_Scan (scan de ports)
    - DDoS_LOIT (attaque par déni de service distribué)

**info** : Note : le dataset source complet (2GB) est conservé et chargé intégralement par Spark —
le filtrage intervient uniquement en aval pour la phase d'entraînement du modèle.

In [3]:
df.groupBy("label").count().orderBy("count", ascending=False).show(truncate=False)

+-----------------+-------+
|label            |count  |
+-----------------+-------+
|Benign           |1786239|
|DoS_Hulk         |349240 |
|NULL             |170195 |
|Port_Scan        |161323 |
|DDoS_LOIT        |95733  |
|FTP-Patator      |9531   |
|DoS_GoldenEye    |8364   |
|DoS_Slowhttptest |6860   |
|SSH-Patator      |5949   |
|Botnet_ARES      |5508   |
|DoS_Slowloris    |5177   |
|Web_Brute_Force  |2734   |
|Web_XSS          |1358   |
|Web_SQL_Injection|24     |
|Heartbleed       |12     |
+-----------------+-------+



### Application des transformations citées ci-dessus

- Benign → trafic normal
- DoS_Hulk → serveur saturé
- Port_Scan → quelqu'un qui cherche des failles
- DDoS_LOIT → attaque coordonnée

In [7]:
from pyspark.sql import functions as F

# définition des 10 labels qu'on veut garder
labels_conserved = [
    "Benign", "DoS_Hulk", "Port_Scan", "DDoS_LOIT",
    "FTP-Patator", "DoS_GoldenEye", "DoS_Slowhttptest",
    "SSH-Patator", "Botnet_ARES", "DoS_Slowloris"
]

# pour éviter le déséquilibre des classes, nous limitons le trafic normal a 100.000 lignes
benign_df = df.filter(F.col("label") == "Benign").limit(100000)

# pour éviter que les attaques avec peu de lignes soient écrasées nous allons également réduire Dos_Hulk + Port_Scan
dos_hulk_df = df.filter(F.col("label") == "DoS_Hulk").limit(50000)
port_scan_df = df.filter(F.col("label") == "Port_Scan").limit(50000)

# nous prenons toutes les attaques sauf le benign
attacks_df = df.filter(
    (F.col("label") != "Benign") &
    (F.col("label") != "DoS_Hulk") &
    (F.col("label") != "Port_Scan") &
    (F.col("label").isin(labels_conserved))
)

# nous collons les 2 df ensemble, le premier qui est limité à 100.000 et l'autre qui contient les attaques uniquement
df_balanced = benign_df.union(dos_hulk_df).union(port_scan_df).union(attacks_df)

# vérification - affiche le nombre de lignes par label
df_balanced.groupBy("label").count().orderBy("count", ascending=False).show()

+----------------+------+
|           label| count|
+----------------+------+
|          Benign|100000|
|       DDoS_LOIT| 95733|
|        DoS_Hulk| 50000|
|       Port_Scan| 50000|
|     FTP-Patator|  9531|
|   DoS_GoldenEye|  8364|
|DoS_Slowhttptest|  6860|
|     SSH-Patator|  5949|
|     Botnet_ARES|  5508|
|   DoS_Slowloris|  5177|
+----------------+------+



# 3. Exploration & Nettoyage des données